# Does physical-AI post-training make Cosmos-Reason2-2B better than its own base model at spotting impossible motion?

Question: NVIDIA built Cosmos-Reason2-2B from Qwen3-VL-2B-Instruct. Does that post-training help the model tell possible motion from impossible motion? Can it act as a critic that filters synthetic video better than its base model?

Setup: two Tesla T4 GPUs (15 GiB each, Turing, no bf16, so fp16). vLLM serves `nvidia/Cosmos-Reason2-2B` on GPU 0 (port 8201) and `Qwen/Qwen3-VL-2B-Instruct` on GPU 1 (port 8202) at the same time. The Cosmos-Reason2-2B model card says it is post-trained from Qwen3-VL-2B-Instruct with the same architecture. So the two models differ only by the post-training. A small 2D physics simulation renders 120 clips of 3 seconds each: 60 obey physics and 60 break one rule. The simulator gives the exact answer for every clip. Each model sees every clip with the same media form and the same fixed question, at temperature 0.

Why a local 2B model: the hosted `nvidia/cosmos-reason2-8b` is in the NVIDIA API model list, but it returned 404 for this account on 2026-11-24. That is why this notebook runs the 2B model itself. The Cosmos model is gated on Hugging Face. You must accept its license on the model page and add an `HF_TOKEN` secret (Add-ons > Secrets). Without the token, the notebook skips Cosmos and still runs the base model.

Exam objectives: physical.3 (Physical AI model families), physical.2 (synthetic data), models.4 (vision-language models), customization.3 (what post-training changes).

Terms used below:

- **Physical AI**: AI that senses, reasons about and acts in the physical world. Robots, vehicles and factories are examples.
- **Cosmos Predict, Transfer, Reason**: three Cosmos model types, split by job. Predict generates future video. Transfer restyles video from control maps. Reason is a vision-language model that reads video and writes text.
- **Vision-language model (VLM)**: a model that takes images or video plus text and writes text.
- **Base model and post-training**: a base model is the starting checkpoint. Post-training means more training on top of it, for example supervised fine-tuning (SFT) or reinforcement learning, with data for one domain. Qwen3-VL-2B-Instruct is the base. Cosmos-Reason2-2B is the post-trained result.
- **Generative and discriminative**: a generative model creates content, such as a video. A discriminative model gives a label to an input. A VLM writes text. When you force a fixed answer set, you use it as a discriminative judge.
- **Synthetic data and ground truth**: synthetic data comes from a simulator or a generator. Ground truth is the known right label. A simulator gives it for free.
- **Critic**: a model that judges generated data and approves or rejects each item.
- **Purity, yield, false rejection**: after filtering, purity is the share of kept clips that are truly good. Yield is the share of all clips that you keep. A false rejection is a good clip that the critic throws away.
- **Precision and recall for "impossible"**: precision is the share of "impossible" answers that are right. Recall is the share of impossible clips that the model finds.
- **Wilson interval**: a 95% range for a share. A small gap between two models inside this range means little.
- **McNemar test (exact)**: a test for two models on the same clips. It looks only at clips where exactly one model is right. If they split about 50/50, the gap is noise.
- **Answer bias**: a model that gives the same kind of answer whatever it sees, or whose answer changes when you ask the same thing in other words.
- **Reasoning format**: when the prompt asks for it, Cosmos Reason writes a `<think>` block first and the final answer after it.
- **fp16 and bf16**: two 16-bit number formats. A T4 runs fp16. It has no bf16.

## Predict before running

Write your own guess first. Then compare it with the tables at the end.

| Question | What you measure | Your guess |
|---|---|---|
| Is Cosmos-Reason2-2B better than its base model at telling possible from impossible motion? | Accuracy with 95% Wilson intervals, on 120 clips | |
| Is the gap bigger than noise? | Clips only one model gets right, exact McNemar p-value | |
| Does the `<think>` format help the 2B models, and what does it cost? | Accuracy, cut replies, tokens and latency, with and without it | |
| Which model leans towards one answer whatever it sees? | Share of "plausible" answers, precision and recall for "impossible" | |
| Does the verdict change when you ask the opposite question? | Flip rate between "plausible or impossible?" and "does it break physics, yes or no?" | |
| Which violation types are easy and which are hard? | Detection rate per violation type | |
| Can either model act as a data critic? | Purity, yield and false rejections on clips not used to choose the mode | |
| Does the model say why? | Share of correct "impossible" answers whose evidence names the right violation | |
| Do both servers take the same media form? | Probe table | |

In [ ]:
# 1. Install vLLM (into the Kaggle VM, not your laptop). imageio-ffmpeg is a fallback video encoder.
# Do not import torch with CUDA in this kernel. It would hold GPU memory that vLLM needs.
!pip install -q vllm imageio-ffmpeg 2>&1 | tail -3
!python -c "import vllm; print('vllm', vllm.__version__)"
!nvidia-smi --query-gpu=index,name,memory.total,compute_cap --format=csv

In [ ]:
# 2. Setup: constants and the Hugging Face token. Standard library, numpy, PIL and pandas. No torch in this kernel.
import atexit, base64, http.client, io, json, math, os, random, re, shutil, statistics, subprocess, time, urllib.error, urllib.request
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
from PIL import Image, ImageDraw, ImageFont

COSMOS, BASE = "nvidia/Cosmos-Reason2-2B", "Qwen/Qwen3-VL-2B-Instruct"
PORTS, GPUS = {COSMOS: 8201, BASE: 8202}, {COSMOS: 0, BASE: 1}   # one model per GPU, both served at the same time
SEED, N_CLIPS = 42, 120
# Context length: a prompt with 12 frames or one video is about 1 to 2 thousand tokens, and a thinking reply can take 3072 more.
# 8192 covers both with room to spare. A longer window would only waste KV cache.
MAX_LEN = 8192
PLAIN_TOKENS, THINK_TOKENS = 800, 3072   # a reply that hits the limit is counted as its own outcome, "cut"
REQ_TIMEOUT_S, MAX_RETRIES, BACKOFF_S = 300, 2, 2.0
CONCURRENCY = 8                          # requests in flight per server. The servers use --max-num-seqs 8 too.
RUN_BUDGET_S, THINK_BUDGET_S = 50 * 60, 40 * 60   # time caps for all runs and for one think run
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
PREFIX = f"{OUT_DIR}/cosmos_reason2_vs_base_t4"
short = lambda m: m.split("/")[-1]

def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN") or None
    except Exception:
        return os.environ.get("HF_TOKEN") or None

HF_TOKEN = get_hf_token()
scrub = lambda s: s.replace(HF_TOKEN, "[hidden]") if HF_TOKEN else s   # use on every log text before you print it
print("HF_TOKEN found:", bool(HF_TOKEN))   # never print the token itself

In [ ]:
# 3. Helpers: one chat call with timeout, retry and backoff; parallel map; answer parsers; Wilson interval; score table
def chat(model, messages, extra=None, max_tokens=300, deadline=None):
    body = {"model": model, "messages": messages, "max_tokens": max_tokens, "temperature": 0, "seed": SEED, **(extra or {})}
    data, url = json.dumps(body).encode(), f"http://localhost:{PORTS[model]}/v1/chat/completions"
    out = dict(ok=False, error="", detail="", content="", finish="", prompt_tokens=0, completion_tokens=0, latency_s=float("nan"), retries=0)
    for attempt in range(MAX_RETRIES + 1):
        timeout = REQ_TIMEOUT_S if deadline is None else max(1, min(REQ_TIMEOUT_S, deadline - time.time()))
        req = urllib.request.Request(url, data, {"Content-Type": "application/json"})
        t0 = time.time()
        try:
            with urllib.request.urlopen(req, timeout=timeout) as r:
                j = json.loads(r.read())
            ch = j["choices"][0]; u = j.get("usage") or {}
            out.update(ok=True, error="", detail="", content=ch["message"].get("content") or "", finish=ch.get("finish_reason") or "",
                       prompt_tokens=u.get("prompt_tokens") or 0, completion_tokens=u.get("completion_tokens") or 0, latency_s=time.time() - t0)
            return out
        except urllib.error.HTTPError as e:           # 429 and 5xx are retried, other codes are final
            out["error"] = f"HTTP {e.code}"
            try: out["detail"] = e.read(300).decode("utf-8", "replace")
            except Exception: pass
            if e.code != 429 and e.code < 500:
                return out
        except (ValueError, KeyError, IndexError, TypeError, AttributeError) as e:   # malformed reply
            out["error"] = "bad reply " + type(e).__name__
            return out
        except (OSError, http.client.HTTPException) as e:   # timeout, refused, reset, cut-off body
            out["error"] = type(getattr(e, "reason", e)).__name__
        wait = min(BACKOFF_S * 2 ** attempt, 30.0)
        if attempt == MAX_RETRIES or (deadline is not None and time.time() + wait >= deadline):
            break
        time.sleep(wait); out["retries"] += 1
    return out

def pmap(fn, items, deadline):   # at most CONCURRENCY calls at once; items not started by the deadline give None
    with ThreadPoolExecutor(CONCURRENCY) as ex:
        return list(ex.map(lambda x: fn(x) if time.time() < deadline else None, items))

def answer_text(text):
    """The final answer of a reply: after the last </think>, inside <answer> if present. Unfinished thinking gives ''."""
    if "</think>" in text:
        text = text.rsplit("</think>", 1)[1]
    elif "<think>" in text:
        return ""
    m = re.search(r"<answer>(.*?)(?:</answer>|$)", text, re.S)
    return (m.group(1) if m else text).strip()

VERDICT = re.compile(r"(?:\bnot|n't) +(?:physically +)?(?:plausible|possible)\b|\b(?:implausible|impossible|unphysical)\b|\b(?:plausible|possible)\b")
def parse_verdict(text):
    """'plausible', 'impossible' or None (empty reply, unfinished thinking, or no verdict word). First verdict word wins."""
    a = answer_text(text).lower()
    m = VERDICT.search(a)
    if m: return "impossible" if m.group(0)[:2] in ("no", "n'", "im", "un") else "plausible"
    m = re.match(r"\W*(yes|no)\b", a)   # a short yes or no to "Is the motion physically possible?"
    return None if not m else "plausible" if m.group(1) == "yes" else "impossible"

def parse_yesno(text):
    """For the opposite question 'Does this video break a law of physics?': yes means impossible, no means plausible."""
    m = re.search(r"\b(yes|no)\b", answer_text(text).lower())
    return None if not m else "impossible" if m.group(1) == "yes" else "plausible"

def wilson(k, n, z=1.96):   # 95% interval for k successes in n tries
    if n == 0:
        return float("nan"), float("nan")
    p, d = k / n, 1 + z * z / n
    c, h = (p + z * z / (2 * n)) / d, z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

def score(g, n_total):   # g: rows of one model and mode. Errors, cut and unparsed replies are left out of accuracy.
    p = g[g.outcome.isin(["correct", "wrong"])]
    n, k = len(p), int((p.outcome == "correct").sum())
    said = int((p.verdict == "impossible").sum()); pos = int((p.label == "impossible").sum())
    tp = int(((p.verdict == "impossible") & (p.label == "impossible")).sum())
    neg = n - pos; lo, hi = wilson(k, n)
    answered = g[g.outcome != "error"]
    nan = float("nan")
    return dict(n=n, errors=int((g.outcome == "error").sum()), cut=int((g.outcome == "cut").sum()), parse_fail=int((g.outcome == "unparsed").sum()),
                accuracy=k / n if n else nan, ci95=f"{lo:.2f}-{hi:.2f}", acc_all=k / n_total,
                precision_imp=tp / said if said else nan, recall_imp=tp / pos if pos else nan,
                false_alarm=(said - tp) / neg if neg else nan, plausible_share=(p.verdict == "plausible").mean() if n else nan,
                p50_s=answered.latency_s.median(), avg_tokens=answered.completion_tokens.mean())

## The clips

A small 2D simulation makes every clip, so the right answer is exact. Units are pixels and seconds. The picture is 320 by 240 pixels, 12 frames per second, 36 frames (3 seconds). Each clip has a black floor line, a timestamp at the top left and simple coloured shapes. Local calls cost nothing, so this notebook uses 120 clips and not 24.

Five scene types. Each scene gives 12 plausible clips and 12 impossible clips. Impossible clips cycle through the violation types of that scene:

| Scene | What happens in a plausible clip | Violation types used in impossible clips |
|---|---|---|
| `drop` | A ball falls and bounces. Each bounce is lower. | `gravity_reversal`, `energy_gain` |
| `roll` | A ball rolls off a table, falls and bounces. | `teleport`, `vanish_return`, `pass_through` |
| `pendulum` | A pendulum swings. Each swing is a little lower. | `amplitude_growth`, `vanish_return` |
| `collide` | A ball hits a resting ball. Momentum is conserved and energy does not grow. | `pass_through`, `spontaneous_motion`, `energy_gain` |
| `push` | A block hits a resting block. The pushed block slides and stops. | `spontaneous_motion`, `teleport` |

That gives 60 plausible and 60 impossible clips with all 7 violation types. Start values (positions, speeds, bounce factors, masses) come from a fixed seed, one per clip. Before any clip goes to a model, the clip cell checks the ground truth from the simulated numbers. In plausible clips, energy never grows, nothing vanishes, jumps or sinks, and a resting object moves only after it is hit. In each impossible clip, the claimed rule really breaks at the claimed frame. If a random draw fails the check, the cell draws again with the next seed and counts the redraws.

In [ ]:
# 4. Physics (same code as cosmos-physical-check): five scenes, exact trajectories, rendering. Units: pixels and seconds, y points down.
W, H, FLOOR, G, RB = 320, 240, 190, 900.0, 12
FPS, NF, SUB = 12, 36, 40                 # 36 frames = 3 s, 40 physics steps per frame
DT = 1 / (FPS * SUB)
TABLE_X, TABLE_Y = 150, 110
TOP = FLOOR - RB                          # centre height of a ball that rests on the floor
RED, GREEN, BLUE, ORANGE, BROWN, GRAY, DARK = (215, 40, 40), (30, 150, 60), (40, 80, 210), (235, 140, 20), (125, 85, 50), (140, 140, 140), (30, 30, 30)
frame_of = lambda k: -(-(k + 1) // SUB)   # first frame that shows the effect of physics step k

def new(scene, viol):
    return dict(scene=scene, viol=viol, label="impossible" if viol else "plausible", vf=None, nv=0,
                E=[], ang=[], pos={}, vis={}, shapes=[], half={})

def snap(c, shapes, **bodies):            # one frame: shapes to draw, and name -> (x, y, visible) for the checks
    c["shapes"].append([s for s in shapes if s])
    for n, (x, y, v) in bodies.items():
        c["pos"].setdefault(n, []).append((x, y)); c["vis"].setdefault(n, []).append(v)

def ball_step(b, g, e, solid):
    """Exact constant-acceleration step with a floor bounce of restitution e. Returns True when the ball reaches the floor."""
    y1 = b["y"] + b["vy"] * DT + 0.5 * g * DT * DT
    touch = g > 0 and b["y"] <= TOP <= y1
    if touch and solid:   # impact speed from energy, so a bounce with e <= 1 never adds energy
        b["y"], b["vy"] = TOP, -e * (b["vy"] ** 2 + 2 * g * (TOP - b["y"])) ** 0.5
    else:
        b["y"], b["vy"] = y1, b["vy"] + g * DT
    b["x"] += b["vx"] * DT
    if not RB <= b["x"] <= W - RB:        # the frame edge acts as a wall
        b["vx"] = -b["vx"]; b["x"] = min(max(b["x"], RB), W - RB)
    return touch

def sim_ball(rng, scene, viol):           # scenes drop and roll
    c, roll = new(scene, viol), scene == "roll"
    b = dict(x=rng.uniform(40, 70), y=TABLE_Y - RB, vx=rng.uniform(80, 110), vy=0.0) if roll else \
        dict(x=rng.uniform(80, 240), y=rng.uniform(30, 50), vx=rng.uniform(-30, 30), vy=0.0)
    e, g, leave = rng.uniform(0.55, 0.75), G, None
    if viol == "energy_gain":
        e, b["y"] = 1.15, 100.0
    for k in range(NF * SUB):
        if k % SUB == 0:
            hide = viol == "vanish_return" and leave is not None and c["vf"] <= k // SUB < c["vf"] + c["nv"]
            snap(c, ([("r", BROWN, 0, TABLE_Y, TABLE_X, FLOOR)] if roll else []) + [None if hide else ("c", RED, b["x"], b["y"], RB)],
                 ball=(b["x"], b["y"], not hide))
        if viol == "gravity_reversal" and k == 4 * SUB:
            g, c["vf"] = -G, frame_of(k)
        if roll and leave is None:        # rolls on the table and slows a little
            b["vx"] = max(0.0, b["vx"] - 10 * DT); b["x"] += b["vx"] * DT
            if b["x"] >= TABLE_X:
                leave = k
                if viol == "vanish_return":
                    c["vf"], c["nv"] = frame_of(k) + 2, 5
        else:
            if viol == "teleport" and k == leave + 82:
                b["x"] += 60; b["y"] -= 70; c["vf"] = frame_of(k)
            if ball_step(b, g, e, viol != "pass_through") and c["vf"] is None and viol in ("pass_through", "energy_gain"):
                c["vf"] = frame_of(k)
        c["E"].append(0.5 * (b["vx"] ** 2 + b["vy"] ** 2) + G * (TOP - b["y"]))
    return c

def rk4(f, s, h):
    a = f(s); b = f(s + h / 2 * a); c = f(s + h / 2 * b); d = f(s + h * c)
    return s + h / 6 * (a + 2 * b + 2 * c + d)

def sim_pendulum(rng, scene, viol):
    c, L, px, py = new(scene, viol), 70.0, W / 2, 50
    damp = -0.5 if viol == "amplitude_growth" else rng.uniform(0.1, 0.3)   # negative damping adds energy
    a0 = rng.uniform(0.45, 0.6) if viol == "amplitude_growth" else rng.uniform(0.5, 0.9)
    s = np.array([rng.choice([-1, 1]) * a0, 0.0])                          # angle and angular speed
    if viol == "vanish_return":
        c["vf"], c["nv"] = 14, 5
    acc = lambda s: np.array([s[1], -G / L * math.sin(s[0]) - damp * s[1]])
    for k in range(NF * SUB):
        if k % SUB == 0:
            hide = viol == "vanish_return" and c["vf"] <= k // SUB < c["vf"] + c["nv"]
            bx, by = px + L * math.sin(s[0]), py + L * math.cos(s[0])
            snap(c, [("r", GRAY, px - 40, py - 8, px + 40, py), None if hide else ("l", DARK, px, py, bx, by),
                     None if hide else ("c", RED, bx, by, RB)], bob=(bx, by, not hide))
            c["ang"].append(s[0])
        s = rk4(acc, s, DT)
        c["E"].append(0.5 * L * L * s[1] ** 2 + G * L * (1 - math.cos(s[0])))
    return c

def sim_pair(rng, scene, viol):           # scenes collide (balls) and push (blocks): A moves right and hits resting B
    c, push = new(scene, viol), scene == "push"
    m = [1.0, rng.uniform(0.6, 1.8)]
    h = [20.0, 20.0 * m[1] ** 0.5] if push else [RB, RB * m[1] ** 0.5]     # half widths. A heavier B is bigger.
    mu = 60.0 if push else 80.0                                            # floor friction (deceleration)
    e = 1.6 if viol == "energy_gain" else rng.uniform(0.1, 0.3) if push else rng.uniform(0.7, 1.0)
    x = [rng.uniform(40, 50), rng.uniform(170, 190)] if push else [rng.uniform(30, 40), rng.uniform(160, 170)]
    v = [rng.uniform(150, 190), 0.0]
    col = (BLUE, ORANGE) if push else (RED, GREEN)
    c["half"] = dict(A=h[0], B=h[1])
    shape = lambda i: ("r", col[i], x[i] - h[i], FLOOR - 2 * h[i], x[i] + h[i], FLOOR) if push else ("c", col[i], x[i], FLOOR - h[i], h[i])
    for k in range(NF * SUB):
        if k % SUB == 0:
            snap(c, [shape(0), shape(1)], A=(x[0], FLOOR - h[0], True), B=(x[1], FLOOR - h[1], True))
        if viol == "spontaneous_motion" and k == 3 * SUB:
            v[1], c["vf"] = 60.0 if push else 70.0, frame_of(k)             # B starts to move, nothing touches it
        if viol == "teleport" and k == 4 * SUB:
            x[1] += 80; c["vf"] = frame_of(k)
        if x[1] - x[0] <= h[0] + h[1] and v[0] > v[1]:                      # contact
            if c["vf"] is None and viol in ("pass_through", "energy_gain"):
                c["vf"] = frame_of(k)
            if viol != "pass_through":                                      # 1D collision with restitution e
                v = [((m[0] - e * m[1]) * v[0] + (1 + e) * m[1] * v[1]) / (m[0] + m[1]),
                     ((m[1] - e * m[0]) * v[1] + (1 + e) * m[0] * v[0]) / (m[0] + m[1])]
        for i in (0, 1):
            v[i] = math.copysign(max(0.0, abs(v[i]) - mu * DT), v[i]); x[i] += v[i] * DT
        c["E"].append(sum(0.5 * m[i] * v[i] ** 2 for i in (0, 1)))
    return c

try: FONT = ImageFont.load_default(size=14)
except TypeError: FONT = ImageFont.load_default()

def render(shapes, f):                    # draw at 2x and shrink, so the edges are smooth
    S = 2
    im = Image.new("RGB", (W * S, H * S), (245, 245, 245)); d = ImageDraw.Draw(im)
    d.rectangle([0, FLOOR * S, W * S, H * S], fill=(205, 205, 205))
    d.line([0, FLOOR * S, W * S, FLOOR * S], fill=(0, 0, 0), width=3 * S)
    for kind, colour, *a in shapes:
        a = [t * S for t in a]
        if kind == "c": d.ellipse([a[0] - a[2], a[1] - a[2], a[0] + a[2], a[1] + a[2]], fill=colour, outline=(0, 0, 0), width=S)
        elif kind == "r": d.rectangle(a, fill=colour, outline=(0, 0, 0), width=S)
        else: d.line(a, fill=colour, width=3 * S)
    im = im.resize((W, H), Image.LANCZOS)
    ImageDraw.Draw(im).text((6, 5), f"t={f / FPS:.2f}s", fill=(0, 0, 0), font=FONT)
    return im

def check(c):
    """Ground truth from the simulated numbers. Plausible clips keep every rule. Impossible clips break the claimed rule at the claimed frame."""
    E, f, v, scene = np.array(c["E"]), c["vf"], c["viol"], c["scene"]
    P = {n: np.array(a) for n, a in c["pos"].items()}
    step = lambda n: np.linalg.norm(np.diff(P[n], axis=0), axis=1)     # pixels moved between two frames
    ang = np.abs(c["ang"]); gap = None
    if "B" in P:
        gap = np.abs(P["B"][:, 0] - P["A"][:, 0]) - sum(c["half"].values())   # free space between A and B
    if not v:
        assert (np.diff(E) <= 1e-6 * E[0]).all(), "energy grew"
        assert all(all(x) for x in c["vis"].values()), "an object vanished"
        assert all(step(n).max() < 50 for n in P), "an object jumped"
        if "ball" in P: assert P["ball"][:, 1].max() <= TOP + 1e-6, "a ball sank into the floor"
        if gap is not None:
            moved = np.abs(P["B"][:, 0] - P["B"][0, 0]) > 0.5
            assert gap.min() > -1.5 and moved.any() and gap[moved.argmax() - 1] < 20, "B overlaps A or moved without a hit"
        if scene == "pendulum": assert ang[-12:].max() <= ang[:6].max(), "the swing grew"
        return
    if v == "gravity_reversal":
        y = P["ball"][:, 1]; d2 = lambda i: y[i + 1] - 2 * y[i] + y[i - 1]      # second difference is the acceleration
        assert d2(f - 2) > 4 and all(d2(i) < -4 for i in (f, f + 1, f + 2)) and y[f:f + 4].max() < TOP - 10, "gravity did not flip in mid-air"
    elif v == "energy_gain":
        dE = np.diff(E); i = int(np.argmax(dE > 0.05 * E[0]))
        assert dE[i] > 0.05 * E[0] and frame_of(i + 1) == f, "energy did not jump at the claimed frame"
    elif v == "pass_through" and scene == "collide":
        assert gap[f:].min() < -0.5 * sum(c["half"].values()), "the balls did not overlap"
    elif v == "pass_through":
        y = P["ball"][:, 1]; assert y[f - 1] <= TOP + 1e-6 and y[f + 2] > TOP + 20, "the ball did not sink into the floor"
    elif v == "vanish_return":
        h = np.array(c["vis"]["bob" if scene == "pendulum" else "ball"])
        assert c["nv"] >= 3 and h[f - 1] and not h[f:f + c["nv"]].any() and h[f + c["nv"]], "no vanish and return"
    elif v == "teleport":
        s = step("ball" if "ball" in P else "B"); assert s[f - 1] > 70 and np.delete(s, f - 1).max() < 50, "no single-frame jump"
    elif v == "amplitude_growth":
        assert ang[-12:].max() > 1.4 * ang[:6].max(), "the swing did not grow"
    elif v == "spontaneous_motion":
        assert gap[f - 1] > 20 and abs(P["B"][f - 1, 0] - P["B"][0, 0]) < 0.5 and abs(P["B"][f + 2, 0] - P["B"][f - 1, 0]) > 8, "B did not start moving alone"
    else:
        raise AssertionError("unknown violation " + v)

In [ ]:
# 5. Build 120 clips (60 plausible, 60 impossible), check the ground truth, encode to mp4, make frame images, show a contact sheet
VIOLS = dict(drop=["gravity_reversal", "energy_gain"], roll=["teleport", "vanish_return", "pass_through"], pendulum=["amplitude_growth", "vanish_return"],
             collide=["pass_through", "spontaneous_motion", "energy_gain"], push=["spontaneous_motion", "teleport"])
PLAN = [(s, v) for s, vs in VIOLS.items() for v in [""] * 12 + [vs[i % len(vs)] for i in range(12)]]
SIM = dict(drop=sim_ball, roll=sim_ball, pendulum=sim_pendulum, collide=sim_pair, push=sim_pair)
IMG_IDX = np.linspace(0, NF - 1, 12).round().astype(int)   # 12 frames for the "frames" form (the server allows 12 images)
SHEET_IDX = [0, 6, 12, 18, 24, 30]

def find_encoder():
    exe = shutil.which("ffmpeg")
    if not exe:
        import imageio_ffmpeg; exe = imageio_ffmpeg.get_ffmpeg_exe()
    return exe

def encode_mp4(frames, path):
    for codec in ("libx264", "mpeg4"):   # H.264 first. MPEG-4 part 2 if this ffmpeg build has no libx264.
        cmd = [FFMPEG, "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{W}x{H}", "-r", str(FPS), "-i", "-",
               "-c:v", codec, "-pix_fmt", "yuv420p", "-movflags", "+faststart", path]
        cmd[-1:-1] = ["-crf", "28", "-preset", "veryfast"] if codec == "libx264" else ["-q:v", "8"]
        if subprocess.run(cmd, input=b"".join(f.tobytes() for f in frames), capture_output=True, timeout=120).returncode == 0:
            return
    raise RuntimeError("ffmpeg could not write an mp4")

def jpeg_b64(im):
    buf = io.BytesIO(); im.save(buf, "JPEG", quality=75); return base64.b64encode(buf.getvalue()).decode()

FFMPEG = find_encoder()
os.makedirs(f"{OUT_DIR}/clips", exist_ok=True)
CLIPS, seen, redraws = [], {}, 0
for scene, viol in PLAN:
    i = seen[scene] = seen.get(scene, -1) + 1
    for attempt in range(30):   # a draw whose numbers do not show the claimed effect is dropped and drawn again
        c = SIM[scene](random.Random(f"{SEED}-{scene}-{i}-{attempt}"), scene, viol)
        try: check(c); break
        except (AssertionError, IndexError, TypeError): redraws += 1
    else:
        raise RuntimeError(f"no valid draw for {scene} {viol}")
    c["id"], c["attempt"] = f"{scene}-{i}", attempt
    frames = [render(s, f) for f, s in enumerate(c.pop("shapes"))]
    assert len(frames) == NF
    path = f"{OUT_DIR}/clips/{c['id']}.mp4"
    encode_mp4(frames, path)
    c["b64_video"], c["mp4_kb"] = base64.b64encode(open(path, "rb").read()).decode(), os.path.getsize(path) / 1000
    c["b64_frames"] = [jpeg_b64(frames[j]) for j in IMG_IDX]
    c["thumbs"] = [frames[j] for j in SHEET_IDX]
    CLIPS.append(c)
LABEL = {c["id"]: c["label"] for c in CLIPS}
assert len(CLIPS) == N_CLIPS and sum(c["label"] == "plausible" for c in CLIPS) == N_CLIPS // 2
INFO = pd.DataFrame([dict(id=c["id"], label=c["label"], violation=c["viol"], claimed_frame=c["vf"], redraws=c["attempt"], mp4_kb=round(c["mp4_kb"], 1)) for c in CLIPS])
INFO.to_csv(f"{PREFIX}_clips.csv", index=False)
print(INFO.groupby(["label", "violation"]).size().unstack(0).fillna(0).astype(int).to_string())
print(f"all ground-truth checks passed for {len(CLIPS)} clips ({redraws} draws were redrawn). Mean mp4 size {INFO.mp4_kb.mean():.0f} KB")

pick = [CLIPS[0]] + [next(c for c in CLIPS if (c["scene"], c["viol"]) == sv) for sv in
        [("drop", "gravity_reversal"), ("roll", "teleport"), ("pendulum", "amplitude_growth"), ("collide", "pass_through"), ("push", "spontaneous_motion")]]
tw, th = W // 2, H // 2
sheet = Image.new("RGB", (tw * len(SHEET_IDX), (th + 16) * len(pick)), "white"); d = ImageDraw.Draw(sheet)
for r, c in enumerate(pick):
    d.text((4, r * (th + 16) + 1), f"{c['id']}: {c['label']} {c['viol']}", fill=(0, 0, 0))
    for j, im in enumerate(c["thumbs"]):
        sheet.paste(im.resize((tw, th)), (j * tw, r * (th + 16) + 16))
sheet.save(f"{PREFIX}_contact_sheet.png")
try:
    from IPython.display import display; display(sheet)
except ImportError:
    pass

## Start both servers

Two `vllm serve` processes start at the same time, one per GPU. Flags and why (vLLM docs: [Multimodal Inputs](https://docs.vllm.ai/en/stable/features/multimodal_inputs), [engine arguments](https://docs.vllm.ai/en/stable/configuration/engine_args), [Qwen3-VL recipe](https://docs.vllm.ai/projects/recipes/en/latest/Qwen/Qwen3-VL.html)):

| Flag | Why |
|---|---|
| `--dtype half` | The T4 has no bf16. |
| `--max-model-len 8192` | A prompt takes 1 to 2 thousand tokens and a thinking reply up to 3072 more. A longer window only wastes KV cache. |
| `--limit-mm-per-prompt '{"video": 1, "image": 12}'` | One video or 12 frames per prompt. This is the JSON form of the flag. |
| `--max-num-seqs 8` | The notebook sends at most 8 requests at once to each server. |
| `--generation-config vllm` | Use vLLM defaults and not the sampling defaults in each model's own `generation_config.json`. Both models then get the same neutral settings. |
| `--trust-request-mm-kwargs` | Recent vLLM rejects a per-request `media_io_kwargs` (such as the video `fps`) unless the server starts with this flag. The cell adds it only if `vllm serve --help=all` lists it. |

The clips travel inside each request as base64 text, so `--allowed-local-media-path` is not needed. The token for the gated Cosmos model goes only into the environment of the Cosmos server process. It never goes into a command line, a file or a printed line. If the token is missing, or the license is not accepted, the cell says what to do and the notebook runs the base model alone.

In [ ]:
# 6. Start both vLLM servers at the same time (Cosmos on GPU 0, base on GPU 1) and wait until each one is healthy.
def gpu_used_mib():
    try:
        return [int(x) for x in subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], text=True).split()]
    except Exception:
        return [0]

def has_flag(flag):   # older vLLM builds do not know --trust-request-mm-kwargs
    try:
        return flag in subprocess.run(["vllm", "serve", "--help=all"], capture_output=True, text=True, timeout=180).stdout
    except Exception:
        return False

TRUST = ["--trust-request-mm-kwargs"] if has_flag("--trust-request-mm-kwargs") else []
SRV, LOGF, READY = {}, {}, []

def launch(model):
    LOGF[model] = f"{OUT_DIR}/vllm_{PORTS[model]}.log"
    cmd = ["vllm", "serve", model, "--dtype", "half", "--max-model-len", str(MAX_LEN), "--gpu-memory-utilization", "0.85", "--max-num-seqs", "8",
           "--limit-mm-per-prompt", json.dumps({"video": 1, "image": 12}), "--generation-config", "vllm", "--port", str(PORTS[model]), *TRUST]
    env = {k: v for k, v in os.environ.items() if k not in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN")}
    env["CUDA_VISIBLE_DEVICES"] = str(GPUS[model])
    if model == COSMOS:
        env["HF_TOKEN"] = HF_TOKEN   # the token travels only in this process environment
    print(short(model), "->", " ".join(cmd[1:]))
    SRV[model] = subprocess.Popen(cmd, stdout=open(LOGF[model], "w"), stderr=subprocess.STDOUT, env=env)

def drop(model, why):
    p = SRV.pop(model)
    if p.poll() is None: p.kill()
    print(f"{short(model)} skipped: {why}\n--- log tail ---\n" + scrub(open(LOGF[model]).read()[-3000:]))

def stop_servers():
    for p in SRV.values(): p.terminate()
    for p in SRV.values():
        try: p.wait(timeout=90)
        except subprocess.TimeoutExpired: p.kill(); p.wait()
    for _ in range(60):                     # wait until the GPU memory is free again
        if max(gpu_used_mib()) < 1500: return
        time.sleep(2)
    print("warning: GPU memory not free after 120 s:", gpu_used_mib())
atexit.register(stop_servers)

if len(gpu_used_mib()) < 2:
    print("WARNING: this notebook needs two GPUs (Kaggle GPU T4 x2). Only one is visible.")
if HF_TOKEN:
    launch(COSMOS)
else:
    print("Cosmos skipped: HF_TOKEN not found.\n  1. Accept the license on https://huggingface.co/nvidia/Cosmos-Reason2-2B\n"
          "  2. Add HF_TOKEN in Add-ons > Secrets and switch it on for this notebook.\nThe base model still runs.")
launch(BASE)
t0 = time.time()
while len(READY) < len(SRV) and time.time() - t0 < 900:
    for m, p in list(SRV.items()):
        if m in READY: continue
        try:
            urllib.request.urlopen(f"http://localhost:{PORTS[m]}/health", timeout=5).close(); READY.append(m)
            print(f"{short(m)} ready after {time.time() - t0:.0f} s")
        except Exception:
            if p.poll() is not None:
                gated = m == COSMOS and re.search(r"401|403|gated|restricted|not authorized", open(LOGF[m]).read(), re.I)
                drop(m, "Hugging Face refused the token. Accept the license on https://huggingface.co/nvidia/Cosmos-Reason2-2B and check HF_TOKEN." if gated else "the server exited")
    time.sleep(2)
for m in list(SRV):
    if m not in READY: drop(m, "not ready after 900 s")
LIVE = [m for m in (COSMOS, BASE) if m in READY]
assert LIVE, "no server started, see the log tails above"
print("models live:", [short(m) for m in LIVE])

In [ ]:
# 7. Sanity check: fp16 on a T4 is not a documented precision for this model. Each server must answer a text question and describe a still frame.
def garbage(t):   # empty, mostly symbols, one character repeated, one word repeated, or NaN-like text
    t = t.strip(); core = t.replace(" ", "")
    return len(core) < 2 or sum(ch.isalpha() for ch in core) < 0.5 * len(core) or bool(re.search(r"(.)\1{9,}|\b(\w+)(?: \2\b){5,}|\bnan\b.*\bnan\b", t, re.I))

still = [{"type": "image_url", "image_url": {"url": "data:image/jpeg;base64," + CLIPS[0]["b64_frames"][0]}}, {"type": "text", "text": "Describe this picture in one sentence."}]
for m in LIVE:
    for name, content, want in (("text question", "What is the capital of France? Answer in one word.", ["paris"]), ("still frame", still, ["ball", "circle"])):
        r = chat(m, [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": content}], max_tokens=80, deadline=time.time() + 120)
        text = r["content"].strip().replace("\n", " ")
        bad, miss = garbage(r["content"]), not any(w in r["content"].lower() for w in want)
        print(f"{short(m):22s} {name:14s} {'WARNING' if bad or miss else 'ok     '} {r['error']} {text[:110]!r}")
        if bad or miss:
            print(f"  WARNING: {short(m)} gave {'garbage or an empty reply' if bad else 'no word from ' + str(want)} for the {name}. fp16 may be broken for this model on a T4.")
            print("  --- log tail ---\n" + scrub(open(LOGF[m]).read()[-1500:]))

## Media probe: which form does each server take?

The Cosmos prompt guide ([Cosmos Cookbook](https://nvidia-cosmos.github.io/cosmos-cookbook/getting_started/prompt_guide/reason_guide.html)) says to use the system prompt "You are a helpful assistant.", to put the media before the text, and to add the `<think>` format after your question if you want reasoning. The Cosmos-Reason2 API page ([NIM docs](https://docs.nvidia.com/nim/vision-language-models/latest/examples/cosmos-reason2/api.html)) says the default video sampling is 4 FPS, which matches the training data, and sets it with a top-level `media_io_kwargs: {"video": {"fps": ...}}`. The same page has a synthetic data critic example: it asks the model to approve or reject a generated video and answers with `Approve` or `Reject`. This notebook asks for `plausible` or `impossible` instead, so a verdict has the same shape for every experiment.

The probe sends one clip to each server in three forms and prints what happened. Both models then use the same form: the first one in this table that works on both servers.

| Form | What is sent |
|---|---|
| `video+fps` | The mp4 clip with `media_io_kwargs: {"video": {"fps": 4.0}}` in the request body |
| `video` | The mp4 clip only. The server picks the frames. |
| `frames` | Twelve frames of the clip as `image_url` parts, in time order |

`prompt_tokens` shows how much visual input the model really got. A form with more tokens gave the model more frames. A form works when the API returns a non-empty reply.

In [ ]:
# 8. Prompts, request builder and media probe: one clip per server and form
QUESTIONS = {"plain": "Is the motion physically possible? Answer plausible or impossible and name the evidence.",
             "opposite": "Does this video break a law of physics? Answer yes or no."}
THINK = "\n\nAnswer the question using the following format:\n\n<think>\nYour reasoning.\n</think>\n\nWrite your final answer immediately after the </think> tag."
FORMS = {"video+fps": {"media_io_kwargs": {"video": {"fps": 4.0}}}, "video": {}, "frames": {}}

def request(clip, form, q="plain", think=False):
    """Messages and extra body fields for one clip. The media part comes before the text part."""
    if form.startswith("video"):
        media, lead = [{"type": "video_url", "video_url": {"url": "data:video/mp4;base64," + clip["b64_video"]}}], ""
    else:
        media = [{"type": "image_url", "image_url": {"url": "data:image/jpeg;base64," + b}} for b in clip["b64_frames"]]
        lead = "These images are frames of one video, in time order. "
    return [{"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": media + [{"type": "text", "text": lead + QUESTIONS[q] + (THINK if think else "")}]}], FORMS[form]

PROBE = []
for m in LIVE:
    for form in FORMS:
        msgs, extra = request(CLIPS[0], form)
        r = chat(m, msgs, extra, max_tokens=200, deadline=time.time() + 120)
        PROBE.append(dict(model=short(m), form=form, works=r["ok"] and bool(r["content"].strip()), error=r["error"], prompt_tokens=r["prompt_tokens"],
                          latency_s=round(r["latency_s"], 1), detail=" ".join(r["detail"].split())[:90]))
PROBE = pd.DataFrame(PROBE)
PROBE.to_csv(f"{PREFIX}_probe.csv", index=False)
print(PROBE.to_string(index=False))
works = {(r.model, r.form) for r in PROBE.itertuples() if r.works}
common = next((f for f in FORMS if all((short(m), f) in works for m in LIVE)), None)
FORM = {m: common or next((f for f in FORMS if (short(m), f) in works), None) for m in LIVE}
if common is None and len(LIVE) > 1:
    print("WARNING: no form works on both servers, so the two models see different input.")
print("form used:", {short(m): f or "none worked: skipped" for m, f in FORM.items()})

## Runner

Each model and mode is one config. A config sends the 120 clips in a fixed mixed order, 8 requests at a time, with a time budget. The two models run at the same time, each on its own GPU. If the budget ends first, the config keeps the results it has and the tables show how many clips finished (`n`). Settings are the same for both models: temperature 0, seed 42, the same media form and the same prompts.

| Mode | Prompt | Token limit |
|---|---|---|
| `plain` | "Is the motion physically possible? Answer plausible or impossible and name the evidence." | 800 |
| `think` | The same question plus the reasoning format from the Cosmos prompt guide | 3072 |
| `opposite` | "Does this video break a law of physics? Answer yes or no." (yes counts as impossible) | 800 |

So the four main runs are base plain, base think, Cosmos plain and Cosmos think. The `opposite` runs test answer bias later.

Each reply gets one outcome: `correct`, `wrong`, `cut` (the reply hit the token limit), `unparsed` (empty reply or no verdict word) or `error` (the API call failed after retries). Accuracy uses `correct` and `wrong` only. The column `acc_all` counts `cut`, `unparsed` and `error` as wrong, over all 120 clips. The prompts name no violation type and give no hint about the variants.

In [ ]:
# 9. Run every config. Each model has its own thread, so both GPUs work at the same time. Think runs last because they are the slowest.
ORDER = random.Random(SEED).sample(range(len(CLIPS)), len(CLIPS))   # mixed scenes, so a cut-off run still covers all of them
START = time.time(); END = START + RUN_BUDGET_S
ROWS = []

def run_config(model, mode):
    q, think = ("opposite", False) if mode == "opposite" else ("plain", mode == "think")
    parse, max_tokens, form = (parse_yesno if q == "opposite" else parse_verdict), THINK_TOKENS if think else PLAIN_TOKENS, FORM[model]
    deadline = min(time.time() + (THINK_BUDGET_S if think else 600), END)
    t0, mine, st = time.time(), [], {"ok": 0, "err": 0}
    def one(i):
        if st["ok"] == 0 and st["err"] >= 4: return   # the server fails every call: stop early
        c = CLIPS[i]; msgs, extra = request(c, form, q, think)
        r = chat(model, msgs, extra, max_tokens=max_tokens, deadline=deadline)
        v = parse(r["content"]) if r["ok"] and r["finish"] != "length" else None
        out = "error" if not r["ok"] else "cut" if r["finish"] == "length" else "unparsed" if v is None else "correct" if v == c["label"] else "wrong"
        st["ok" if r["ok"] else "err"] += 1
        row = dict(cfg=short(model) + "/" + mode, model=model, mode=mode, form=form, clip=c["id"], scene=c["scene"], label=c["label"], viol=c["viol"],
                         verdict=v, outcome=out, evidence=answer_text(r["content"])[:400], finish=r["finish"], latency_s=r["latency_s"],
                         completion_tokens=r["completion_tokens"], retries=r["retries"], error=r["error"])
        mine.append(row); ROWS.append(row)
    pmap(one, ORDER, deadline)
    count = lambda o: sum(x["outcome"] == o for x in mine)
    note = "" if len(mine) == len(CLIPS) else f"  (cut off: {len(mine)} of {len(CLIPS)} clips reached)"
    print(f"{short(model):22s} {mode:9s} {form:10s} correct {count('correct'):3d}  wrong {count('wrong'):3d}  cut {count('cut'):3d}  unparsed {count('unparsed'):3d}  "
          f"errors {count('error'):3d}  {time.time() - t0:5.0f}s{note}")

def run_model(m):
    for mode in ("plain", "opposite", "think"):
        run_config(m, mode)

RUN = [m for m in LIVE if FORM.get(m)]
with ThreadPoolExecutor(max(1, len(RUN))) as ex:
    list(ex.map(run_model, RUN))
print(f"runs took {(time.time() - START) / 60:.1f} min")

In [ ]:
# 10. Summary table per model and mode (also written to CSV), with the trivial baseline and 95% Wilson intervals
assert ROWS, "No model answered. Check the probe table and the server logs above."
RAW = pd.DataFrame(ROWS)
RAW.to_csv(f"{PREFIX}_raw.csv", index=False)
lo, hi = wilson(N_CLIPS // 2, N_CLIPS)
SUMMARY = pd.DataFrame(
    [dict(cfg=cfg, model=g.model.iloc[0], mode=g["mode"].iloc[0], **score(g, N_CLIPS)) for cfg, g in RAW.groupby("cfg", sort=False)]
    + [dict(cfg="always plausible", model="always plausible", mode="baseline", n=N_CLIPS, errors=0, cut=0, parse_fail=0, accuracy=0.5, ci95=f"{lo:.2f}-{hi:.2f}",
            acc_all=0.5, precision_imp=float("nan"), recall_imp=0.0, false_alarm=0.0, plausible_share=1.0, p50_s=float("nan"), avg_tokens=float("nan"))])
SUMMARY.round(3).to_csv(f"{PREFIX}.csv", index=False)
print(SUMMARY.drop(columns=["model", "mode"]).round(2).to_string(index=False))

# detection rate per violation type: impossible clips that the model called impossible, as hits/clips (plain and think runs)
imp = RAW[(RAW.label == "impossible") & RAW.outcome.isin(["correct", "wrong"]) & RAW["mode"].isin(["plain", "think"])]
DETECT = imp.assign(hit=imp.verdict == "impossible").groupby(["viol", "cfg"]).hit.agg(lambda x: f"{int(x.sum())}/{len(x)}").unstack("cfg").fillna("-")
DETECT.to_csv(f"{PREFIX}_by_violation.csv")
print("\nImpossible clips found, per violation type (hits/clips):")
print(DETECT.to_string())

## Paired comparison: Cosmos against its base on the same clips

Two accuracy numbers on the same 120 clips are not two independent samples. The same clips are easy or hard for both models. The exact McNemar test uses only the clips where exactly one model is right. Call them `b` (Cosmos right alone) and `c` (base right alone). If neither model is better, each such clip is a fair coin toss. With `n = b + c`, the two-sided p-value is `2 * sum(C(n, i) for i in 0..min(b, c)) / 2^n`, capped at 1. The cell computes it with `math.comb`.

The first row compares the best mode of each model. The best mode is the one with the higher `acc_all` in the summary. It is chosen on the same clips, so this row is a little optimistic. The other rows compare the same mode on both sides. Each pair has two rows. "All clips" counts a reply that was cut or had no verdict as wrong, so a model that only writes longer replies can look worse. "Both answered" uses only the clips where both models gave a verdict. Read both rows.

In [ ]:
# 11. Paired comparison: clips only one model gets right, exact McNemar p-value
def mcnemar_exact(b, c):
    n = b + c
    return 1.0 if n == 0 else min(1.0, 2 * sum(math.comb(n, i) for i in range(min(b, c) + 1)) / 2 ** n)

S = SUMMARY.set_index("cfg")
def best_cfg(model):   # plain or think mode with the higher acc_all (clips without a usable answer count as wrong)
    g = S[(S.model == model) & S["mode"].isin(["plain", "think"])]
    return g.acc_all.idxmax() if len(g) else None

PAIRED = pd.DataFrame()
if len(RUN) < 2:
    print("Both models are needed for a paired comparison. Only", [short(m) for m in RUN], "ran.")
else:
    right = lambda cfg: set(RAW[(RAW.cfg == cfg) & (RAW.outcome == "correct")]["clip"])
    pairs = [("best mode of each", best_cfg(COSMOS), best_cfg(BASE))] + [(f"{md} against {md}", f"{short(COSMOS)}/{md}", f"{short(BASE)}/{md}") for md in ("plain", "opposite", "think")]
    rows = []
    for name, a, b in pairs:
        if a not in S.index or b not in S.index: continue
        usable = lambda cfg: set(RAW[(RAW.cfg == cfg) & RAW.outcome.isin(["correct", "wrong"])]["clip"])
        U = usable(a) & usable(b)   # clips where both gave a verdict
        for scope, keep in (("all clips (no verdict = wrong)", None), ("both answered", U)):
            A, B = right(a) if keep is None else right(a) & keep, right(b) if keep is None else right(b) & keep
            n = N_CLIPS if keep is None else len(keep)
            if n == 0: continue
            x, y = len(A - B), len(B - A)
            rows.append(dict(pair=name, scope=scope, n=n, cosmos_cfg=a, base_cfg=b, cosmos_only=x, base_only=y, both=len(A & B), neither=n - len(A | B),
                             cosmos_acc=len(A) / n, base_acc=len(B) / n, p_exact=mcnemar_exact(x, y)))
    PAIRED = pd.DataFrame(rows)
    PAIRED.round(4).to_csv(f"{PREFIX}_paired.csv", index=False)
    print(PAIRED.round(4).to_string(index=False))
    for r in PAIRED.itertuples():
        lead = "Cosmos" if r.cosmos_only > r.base_only else "The base model" if r.base_only > r.cosmos_only else "Neither model"
        print(f"{r.pair}, {r.scope}: Cosmos is right alone on {r.cosmos_only} clips, the base on {r.base_only}. Exact McNemar p = {r.p_exact:.4f}. " +
              (f"The gap is inside the noise. {lead} leads on paper, but {r.n} clips cannot tell the two models apart." if r.p_exact >= 0.05
               else f"{lead} is ahead, and chance alone is an unlikely cause (p below 0.05)."))

## Prompt sensitivity: does the verdict change when you ask the opposite question?

A model that really judges the motion should give the same verdict when you ask the same thing in other words. The `plain` question asks "plausible or impossible?". The `opposite` question asks "does this video break a law of physics? Answer yes or no." Yes means impossible and no means plausible. The cell compares the two verdicts on the clips where both replies gave a verdict. A flip is a clip where the two verdicts differ. A high flip rate means the answer depends on the wording, which is answer bias. A model can flip a lot and still have a decent accuracy in one wording, so read the share of "impossible" answers under each wording too.

In [ ]:
# 12. Prompt sensitivity: same clips, opposite question, plain mode, both models
FLIP = []
for m in RUN:
    a = RAW[(RAW.model == m) & (RAW["mode"] == "plain")].set_index("clip").verdict.to_frame("plain")
    b = RAW[(RAW.model == m) & (RAW["mode"] == "opposite")].set_index("clip").verdict.to_frame("opposite")
    j = a.join(b, how="inner").dropna()
    if j.empty: continue
    lab = j.index.map(LABEL)
    FLIP.append(dict(model=short(m), n_both=len(j), flips=int((j.plain != j.opposite).sum()), flip_rate=(j.plain != j.opposite).mean(),
                     impossible_share_plain=(j.plain == "impossible").mean(), impossible_share_opposite=(j.opposite == "impossible").mean(),
                     acc_plain=(j.plain == lab).mean(), acc_opposite=(j.opposite == lab).mean(), right_in_both=((j.plain == lab) & (j.opposite == lab)).mean()))
FLIP = pd.DataFrame(FLIP)
FLIP.round(3).to_csv(f"{PREFIX}_prompt_flip.csv", index=False)
print(FLIP.round(3).to_string(index=False) if len(FLIP) else "No clip got a verdict under both wordings.")

## Critic: filter a synthetic set

A data pipeline that makes video with a generator keeps some bad clips. Here the critic reads each clip and keeps it only if the verdict is `plausible`. A clip goes if the verdict is `impossible`, or if the model gave no usable answer.

To avoid choosing and measuring on the same clips, the 120 clips split into two halves of 60 (30 plausible and 30 impossible each), with a fixed seed. For each model, the cell picks the mode (`plain` or `think`) with the higher `acc_all` on half A. Then it measures the critic on half B only. On half B it builds 200 random sets of 40 clips with a known 30% of bad clips (28 plausible and 12 impossible), and it reports the mean of each number over those sets.

A filter is not a fix. It removes bad clips and does not repair them. Every false rejection is a good clip that you lose. Every miss is a bad clip that stays in the data.

In [ ]:
# 13. Critic: choose the mode on half A, measure purity, yield and false rejections on half B, for sets with a known 30% of bad clips
def critic_eval(verdict, good, bad, rng, draws=200, n_good=28, n_bad=12):
    """Mean over random sets. A clip stays only when its verdict is plausible. Clips with no verdict go."""
    acc = {k: [] for k in ("purity_after", "yield_kept", "false_rejection_rate", "bad_removed_rate")}
    for _ in range(draws):
        gk = sum(verdict.get(c) == "plausible" for c in rng.sample(good, n_good)); bk = sum(verdict.get(c) == "plausible" for c in rng.sample(bad, n_bad))
        acc["purity_after"].append(gk / (gk + bk) if gk + bk else float("nan")); acc["yield_kept"].append((gk + bk) / (n_good + n_bad))
        acc["false_rejection_rate"].append(1 - gk / n_good); acc["bad_removed_rate"].append(1 - bk / n_bad)
    return {k: float(pd.Series(v).mean()) for k, v in acc.items()}   # NaN sets (nothing kept) are skipped

rng = random.Random(SEED)
good = [c["id"] for c in CLIPS if c["label"] == "plausible"]; bad = [c["id"] for c in CLIPS if c["label"] == "impossible"]
rng.shuffle(good); rng.shuffle(bad)
HALF_A, B_GOOD, B_BAD = good[:30] + bad[:30], good[30:], bad[30:]
CRITIC = []
for m in RUN:
    acc_a = {md: (RAW[(RAW.cfg == f"{short(m)}/{md}") & RAW["clip"].isin(HALF_A)].outcome == "correct").sum() / len(HALF_A) for md in ("plain", "think")}
    md_ = max(acc_a, key=lambda k: (acc_a[k], k == "plain"))   # a tie goes to plain, which is cheaper
    verdict = RAW[RAW.cfg == f"{short(m)}/{md_}"].set_index("clip").verdict.to_dict()
    CRITIC.append(dict(model=short(m), mode_chosen_on_A=md_, acc_all_A=acc_a[md_], purity_before=0.7, **critic_eval(verdict, B_GOOD, B_BAD, random.Random(SEED))))
CRITIC = pd.DataFrame(CRITIC)
CRITIC.round(3).to_csv(f"{PREFIX}_critic.csv", index=False)
print(CRITIC.round(3).to_string(index=False))
print("purity_before is 0.7 by construction. A filter is not a fix: rejected clips are lost, not repaired, and every kept bad clip is a miss.")

## Grounding check: does the evidence name the right violation?

A correct "impossible" answer can still be a guess. The next cell looks at the answers where a model said "impossible" for a clip that really is impossible (plain and think runs). It reads the stated evidence (the final answer, not the thinking block) and looks for words that fit the true violation type. For example `teleport` looks for words such as "jump", "suddenly" and "instant". The word lists are short and made by hand. This is a rough check. A hit does not prove that the model understood the clip, and a miss can be a good answer in other words. The column `names_any_violation` shows how often a reply hits any list at all, so you can see how loose the rubric is.

In [ ]:
# 14. Grounding check: keyword rubric on the stated evidence of correct "impossible" answers (rough check)
KEYS = {
    "gravity_reversal": ["gravity", "upward", "rises", "rising", "float", "against", "reverse", "fly", "flies", "defies"],
    "pass_through": ["through", "penetrat", "overlap", "phase", "sink", "sinks", "ghost", "clip", "no collision", "without colliding", "without bouncing"],
    "vanish_return": ["vanish", "disappear", "reappear", "invisible", "missing", "gone", "permanence", "pops"],
    "energy_gain": ["energy", "higher than", "faster than", "gains", "gain speed", "speeds up", "more speed", "higher each", "higher after"],
    "amplitude_growth": ["amplitude", "swing", "larger", "grow", "increas", "higher each", "wider"],
    "teleport": ["teleport", "jump", "suddenly", "instant", "discontinu", "skip", "abrupt"],
    "spontaneous_motion": ["no cause", "without", "spontaneous", "on its own", "by itself", "untouched", "no contact", "not touched", "no force", "nothing"],
}
hit = lambda text, t: any(w in text.lower() for w in KEYS[t])
gr = RAW[(RAW.label == "impossible") & (RAW.verdict == "impossible") & RAW["mode"].isin(["plain", "think"])]
gr = gr.assign(right=[hit(e, t) for e, t in zip(gr.evidence, gr.viol)], any_=[any(hit(e, t) for t in KEYS) for e in gr.evidence])
GROUND = gr.groupby("cfg").agg(correct_impossible=("clip", "size"), names_right_violation=("right", "sum"), names_any_violation=("any_", "sum"))
GROUND["right_rate"] = GROUND.names_right_violation / GROUND.correct_impossible
GROUND.round(2).to_csv(f"{PREFIX}_grounding.csv")
print(GROUND.round(2).to_string() if len(GROUND) else "No model gave a correct 'impossible' answer, so there is no evidence to check.")
for _, r in gr.head(6).iterrows():
    print(f"[{r.cfg} | {r.viol} | right={r.right}]", r.evidence[:160].replace("\n", " "))

## Cosmos and GR00T: what each takes in and gives out

The facts below come from the NVIDIA documentation, blog and Hugging Face pages listed in the `source` column. The last row is about hardware. It explains why this notebook runs a 2B Cosmos Reason model on a T4 and does not try the newer Cosmos 3 models.

In [ ]:
# 15. Facts table: Cosmos model types, Isaac GR00T, and what a T4 can run
FACTS = pd.DataFrame([
    dict(family="Cosmos Predict", job="Generates future world states as video", takes_in="Text, image or video (also start and end frames)",
         gives_out="Video", used_here="no",
         source="NVIDIA technical blog 'Scale Synthetic Data and Physical AI Reasoning with NVIDIA Cosmos World Foundation Models'; NIM for Cosmos WFM docs (text2video, image2video, video2video)"),
    dict(family="Cosmos Transfer", job="Restyles a video and keeps its structure (for example sim to real)", takes_in="Control video (edge, blur, depth, segmentation or world-space map) plus a text prompt",
         gives_out="Photorealistic video with the same layout and motion", used_here="no",
         source="NIM for Cosmos WFM docs, Transfer page; saved reading cosmos.txt (Omniverse simulation into Cosmos Transfer)"),
    dict(family="Cosmos Reason", job="Reasons about physical scenes. Reads video and writes text. Can act as a critic.", takes_in="Video or image plus a text question",
         gives_out="Text, with an optional <think> block before the answer", used_here="yes: nvidia/Cosmos-Reason2-2B against its base Qwen3-VL-2B-Instruct",
         source="Hugging Face model card nvidia/Cosmos-Reason2-2B (post-trained from Qwen3-VL-2B-Instruct); NIM for VLMs docs, Cosmos Reason2 API page (video_url, <think> format, Synthetic Data Generation Critic); Cosmos Cookbook prompt guide"),
    dict(family="Isaac GR00T", job="Robot policy. A vision-language-action (VLA) model for humanoid and other robots", takes_in="Camera views, a language instruction and the robot's current state",
         gives_out="Chunks of continuous robot actions", used_here="no",
         source="saved reading gr00t.txt (GR00T 1.7 is an open 3B VLA with a Cosmos backend)"),
    dict(family="Cosmos 3 Edge and Nano", job="Omnimodal world models: generate video, image, audio and actions, and understand them", takes_in="Text, image, video and action trajectories",
         gives_out="Text, image, video and actions", used_here="no: a T4 cannot run them (Turing, no bf16)",
         source="Hugging Face model cards nvidia/Cosmos3-Edge and nvidia/Cosmos3-Nano: only BF16 is tested, FP16 is not officially supported. The Edge card lists Ampere, Hopper and Blackwell GPUs. The Nano card lists GB200 and H100 as test hardware."),
])
FACTS.to_csv(f"{PREFIX}_facts.csv", index=False)
for _, r in FACTS.iterrows():
    print(f"{r.family} (used here: {r.used_here})\n  job:       {r.job}\n  takes in:  {r.takes_in}\n  gives out: {r.gives_out}\n  source:    {r.source}\n")

In [ ]:
# 16. Stop both servers and wait until the GPU memory is free
stop_servers()
print("servers stopped. GPU memory in use (MiB):", gpu_used_mib())
print("files in", OUT_DIR + ":", sorted(f for f in os.listdir(OUT_DIR) if f.startswith("cosmos_reason2_vs_base_t4")))

## What to look for

- **Baseline and intervals**: the always-plausible baseline scores 0.5. A model is only clearly better than guessing when its whole 95% Wilson interval sits above 0.5. If the two intervals overlap a lot, read the paired test and not the two numbers.
- **Paired test**: a p-value of 0.05 or more means the gap between Cosmos and its base is inside the noise. A small p-value shows that the gap is not chance. It does not show that the gap is large or that it holds on real video.
- **`plausible_share`, `precision_imp`, `recall_imp`, `false_alarm`**: a share near 1 means the model says "plausible" for almost everything. Its accuracy then tracks the baseline. A share near 0 means it flags almost everything.
- **`cut`, `parse_fail`, `errors`, `acc_all`**: read accuracy together with these. A think run with many cut replies can look good on the clips it finished. `acc_all` counts every unfinished clip as wrong.
- **`plain` against `think`**: compare accuracy with `avg_tokens`, `cut` and `p50_s`. Reasoning costs output tokens and seconds. Check if the gain pays for them, and check it for each model.
- **Prompt sensitivity**: a high `flip_rate` means the verdict follows the wording. Compare `impossible_share_plain` with `impossible_share_opposite` to see which wording pushes the model towards "impossible".
- **Detection per violation type**: each type has 6 to 10 clips. Look for patterns, such as motion that starts without a cause against a bounce with too much energy. One type is a hint, not a rate.
- **Critic**: compare `purity_after` with `purity_before`, then look at `yield_kept` and `false_rejection_rate`. A higher purity that costs most of the good clips may not be worth it. The mode was chosen on half A and measured on half B.
- **Grounding**: compare `names_right_violation` with `names_any_violation`. If the second is close to the first, the word lists are loose and the check says little.
- **Probe table**: `prompt_tokens` shows how many frames the model really received for each form.
- **Limits**: the clips are simple 2D shapes, not real or generated video. The question is fixed. One run at temperature 0 gives one answer per clip. NVIDIA recommends temperature 0.6 to 0.7 for these models, so greedy decoding may differ from the best setting. fp16 on a T4 is not a setting NVIDIA tested for vLLM serving. The sanity cell checks only for obvious breakage.

## Exam rules this proves

1. **customization.3**: Post-training changes a base model's weights for one domain. To see what it changed, compare it with its own base on the same inputs, the same prompt and the same settings.
2. **models.4**: A VLM becomes a discriminative judge when you force a fixed answer set. Ask the same thing in opposite words to check for answer bias.
3. **physical.2**: A simulator gives exact ground truth for synthetic data. Check the truth before you trust it, then use it to measure a critic.
4. **physical.2**: A critic raises purity and lowers yield, and it can reject good items. Choose its settings on one half of the data and measure on the other half.
5. **physical.3**: Cosmos Reason is the Cosmos model type that understands video. Cosmos 3 Edge and Nano need Ampere or newer GPUs with BF16, so check the GPU architecture before you pick a model.